In [213]:
from pyoxigraph import Store
from tabulate import tabulate

In [214]:
filename = "dataset.ttl"

In [215]:
text1 = '''CQ_001
Given a publication PID, find all articles that cite it plus those cited in the publication itself.'''

query1 = '''
PREFIX cito: <http://purl.org/spar/cito/>
PREFIX datacite: <http://purl.org/spar/datacite/>
PREFIX fabio: <http://purl.org/spar/fabio/>
PREFIX literal: <http://www.essepuntato.it/2010/06/literalreification/>

SELECT DISTINCT ?pid_value
WHERE {
    VALUES ?article_id_value {"10.1007/978-3-319-65633-5"}
    ?article datacite:hasIdentifier [
        literal:hasLiteralValue ?article_id_value
    ] .
    {
        ?article cito:cites [
            a fabio:ScholarlyWork ;
            datacite:hasIdentifier [
                literal:hasLiteralValue ?pid_value
            ]
        ] .
    }
    UNION
    {
    ?article_citing a fabio:ScholarlyWork ;
        cito:cites ?article ;
        datacite:hasIdentifier [
            literal:hasLiteralValue ?pid_value
        ] .
    }
}
'''

In [216]:
text2 = '''CQ_002
Given an author, return all documents where one of his/her work is cited.
'''

query2 = '''
PREFIX cito: <http://purl.org/spar/cito/>
PREFIX datacite: <http://purl.org/spar/datacite/>
PREFIX fabio: <http://purl.org/spar/fabio/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX literal: <http://www.essepuntato.it/2010/06/literalreification/>
PREFIX pro: <http://purl.org/spar/pro/>

SELECT DISTINCT ?pid_value
WHERE {
    VALUES ?author_id_value {"0000-0001-5145-0843"}
    ?author datacite:hasIdentifier [
        literal:hasLiteralValue ?author_id_value ;
        datacite:usesIdentifierScheme datacite:orcid
    ] ;
    ^pro:isHeldBy [
        pro:withRole pro:author ;
        ^pro:isRelatedToRoleInTime [
            ^cito:cites [
                datacite:hasIdentifier [
                    literal:hasLiteralValue ?pid_value
                ]
             ]
         ]
    ] .
}
'''

In [217]:
text3 = '''CQ_003
Give me the list of identifiers of all the available datasets related to topic X.
'''

query3 = '''
PREFIX bido: <http://purl.org/spar/bido/>
PREFIX datacite: <http://purl.org/spar/datacite/>
PREFIX fabio: <http://purl.org/spar/fabio/>
PREFIX literal: <http://www.essepuntato.it/2010/06/literalreification/>
PREFIX prism: <http://prismstandard.org/namespaces/basic/2.0/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT DISTINCT ?pid_value
WHERE {
    VALUES ?topic_label {"research"}
    ?dataset a fabio:Dataset ;
        bido:holdsBibliometricDataInTime [
            bido:withBibliometricData [
                skos:prefLabel ?topic_label
            ]
        ] ;
        datacite:hasIdentifier [
            datacite:usesIdentifierScheme datacite:doi ;
            literal:hasLiteralValue ?pid_value
        ] .
}
'''

In [218]:
text7 = '''CQ_007
Given the PID of a publication, I want to align its bibliographic metadata across
graphs, by also removing duplicated elements.
'''

query7 = r'''
PREFIX dcterms:  <http://purl.org/dc/terms/>
PREFIX datacite: <http://purl.org/spar/datacite/>
PREFIX literal:  <http://www.essepuntato.it/2010/06/literalreification/>
PREFIX fabio:    <http://purl.org/spar/fabio/>
PREFIX frbr:     <http://purl.org/vocab/frbr/core#>
PREFIX pro:      <http://purl.org/spar/pro/>

SELECT DISTINCT ?field ?canonicalValue
WHERE {
  {
    ?id_1 a datacite:Identifier ; literal:hasLiteralValue "10.1007/978-3-319-65633-5" .
    ?work datacite:hasIdentifier ?id_1 ; a fabio:ScholarlyWork .
    
    ?work dcterms:title ?rawValue .
    BIND("title" AS ?field)
    BIND(REPLACE(STR(?rawValue), "^\\s+|\\s+$", "") AS ?canonicalValue)
  }
  UNION
  {
    ?id_2 a datacite:Identifier ; literal:hasLiteralValue "10.1007/978-3-319-65633-5" .
    ?work datacite:hasIdentifier ?id_2 ; a fabio:ScholarlyWork .
    
    ?work dcterms:issued ?rawValue .
    BIND("date" AS ?field)
    BIND(REPLACE(STR(?rawValue), "^\\s+|\\s+$", "") AS ?canonicalValue)
  }
  UNION
  {
    ?id_3 a datacite:Identifier ; literal:hasLiteralValue "10.1007/978-3-319-65633-5" .
    ?work datacite:hasIdentifier ?id_3 ; a fabio:ScholarlyWork .
    
    ?work datacite:hasIdentifier ?identifier .
    ?identifier literal:hasLiteralValue ?rawValue .
    BIND("identifier" AS ?field)
    BIND(REPLACE(STR(?rawValue), "^\\s+|\\s+$", "") AS ?canonicalValue)
  }
  UNION
  {
    ?id_4 a datacite:Identifier ; literal:hasLiteralValue "10.1007/978-3-319-65633-5" .
    ?work datacite:hasIdentifier ?id_4 ; a fabio:ScholarlyWork .
    
    ?work frbr:partOf ?rawValue .
    BIND("isPartOf" AS ?field)
    BIND(STR(?rawValue) AS ?canonicalValue)
  }
  UNION
  {
    ?id_5 a datacite:Identifier ; literal:hasLiteralValue "10.1007/978-3-319-65633-5" .
    ?work datacite:hasIdentifier ?id_5 ; a fabio:ScholarlyWork .
    
    ?work pro:isRelatedToRoleInTime ?roleInTime .
    ?roleInTime pro:withRole pro:author ;
                pro:isHeldBy ?rawValue .

    OPTIONAL {
      ?rawValue datacite:hasIdentifier ?authorId .
      ?authorId datacite:usesIdentifierScheme datacite:orcid ;
                literal:hasLiteralValue ?orcid .
    }

    BIND("creator" AS ?field)
    BIND(COALESCE(CONCAT("orcid:", STR(?orcid)), STR(?rawValue)) AS ?canonicalValue)
  }
}
ORDER BY ?field ?canonicalValue
LIMIT 50
'''

In [219]:
text14 = '''CQ_014
What are the topics and scientific domains covered in these graphs?
'''

query14 = '''
PREFIX bido: <http://purl.org/spar/bido/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT DISTINCT ?topic_label
WHERE {
    ?work bido:holdsBibliometricDataInTime [
         bido:withBibliometricData [
             skos:prefLabel ?topic_label
         ]
    ] .
}
'''

In [220]:
text22 = '''CQ_022
I have to do a research on a specific place, or a specific geographic area, defined
by keywords, and I need dataset for a specific subject, also defined by keywords.
Give me the references of all the available datasets for that place and subject that
are present in all the graphs.
'''

query22 = '''
PREFIX bido: <http://purl.org/spar/bido/>
PREFIX datacite: <http://purl.org/spar/datacite/>
PREFIX fabio: <http://purl.org/spar/fabio/>
PREFIX literal: <http://www.essepuntato.it/2010/06/literalreification/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT DISTINCT ?pid_value
WHERE {
    VALUES ?topic_label {"research"}
    VALUES ?place_label {"india"}
    ?dataset a fabio:Dataset .
     ?dataset bido:holdsBibliometricDataInTime [
            bido:withBibliometricData [
               skos:prefLabel ?topic_label
            ]
     ] .
     ?dataset bido:holdsBibliometricDataInTime [
         bido:withBibliometricData [
            skos:prefLabel ?place_label
            ]
     ] .
     ?dataset datacite:hasIdentifier [
         literal:hasLiteralValue ?pid_value ;
         datacite:usesIdentifierScheme datacite:doi
     ] .
}
'''

In [221]:
text24 = '''CQ_024
I want to know what authors are most key to read in relation to topic x and whom
generally publish in immediate open access.
'''

query24 = '''
PREFIX bido: <http://purl.org/spar/bido/>
PREFIX cito: <http://purl.org/spar/cito/>
PREFIX datacite: <http://purl.org/spar/datacite/>
PREFIX fabio: <http://purl.org/spar/fabio/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX frbr: <http://purl.org/vocab/frbr/core#>
PREFIX literal: <http://www.essepuntato.it/2010/06/literalreification/>
PREFIX pro: <http://purl.org/spar/pro/>
PREFIX pso: <http://purl.org/spar/pso/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT DISTINCT ?author_name (SUM(?citations) AS ?tot_citations)
WHERE {
    {
        SELECT ?article (COUNT(?citing) as ?citations)
        WHERE {
             ?citing cito:cites ?article .
        }
        GROUP BY ?article
    }
    VALUES ?topic_label {"open access"}
    ?article bido:holdsBibliometricDataInTime [
            bido:withBibliometricData [
                skos:prefLabel ?topic_label
            ]
        ] ;
        pro:isRelatedToRoleInTime [
            pro:withRole pro:author ;
            pro:isHeldBy [
                foaf:name ?author_name
            ]
        ] ;
        frbr:realization [
            pso:holdsStatusInTime [
                pso:withStatus pso:open-access
            ]
        ] .
}
GROUP BY ?author_name
ORDER BY DESC(?tot_citations)
LIMIT 10
'''

In [222]:
queries = [(text1, query1),
           (text2, query2),
           (text3, query3),
           (text7, query7),
           (text14, query14),
           (text22, query22),
           (text24, query24),]

store = Store()
store.bulk_load(filename, "text/turtle")

for text, query_text in queries:
    solutions = store.query(query_text)
    variables = solutions.variables

    print(text)
    table = []
    for solution in solutions:
        row = []
        for var in variables:
            term = solution[var]
            if term is None:
                row.append(None)
            elif hasattr(term, "value"):
                row.append(term.value)
            else:
                row.append(str(term))
        table.append(row)
        
    headers = [str(v) for v in variables]
    print(tabulate(table, headers=headers, tablefmt="psql"))

CQ_001
Given a publication PID, find all articles that cite it plus those cited in the publication itself.
+------------------------------+
| ?pid_value                   |
|------------------------------|
| 10.3390/app8122687           |
| 10.3390/rs11222599           |
| 10.3390/data4030092          |
| 10.3390/ijgi10040244         |
| 10.3390/sci6020026           |
| 10.1109/jstars.2021.3134785  |
| 10.1117/12.2576171           |
| 10.3390/app13095646          |
| 10.3390/rs13183763           |
| 10.1007/s41651-022-00130-0   |
| 10.1186/s12302-020-00397-4   |
| 10.3390/rs14153606           |
| 10.3390/rs14020393           |
| 10.31223/osf.io/7zsyr        |
| 10.3390/su122410411          |
| 10.3390/data4030093          |
| 10.3390/w12102783            |
| 10.3390/ijgi7070276          |
| 10.3390/app9071459           |
| 10.3390/rs13163224           |
| 10.1007/978-3-030-77044-0_1  |
| 10.1007/978-981-32-9915-3_5  |
| 10.1016/j.asr.2020.08.026    |
| 10.1785/0220210233           |
| 